# 05. Policy Application

2단계 적정가격 결과에 유류세 인하 등 국내 정책 효과를 반영하고, 최고가격제 구간은 별도 점검합니다.

이 노트북은 Google Colab에서 단독 실행되도록 구성되어 있습니다. 설정 셀의 경로만 본인 Google Drive 구조에 맞게 수정한 뒤 위에서 아래로 실행하면 됩니다.


In [ ]:
# =========================================================
# 0) Colab 실행 설정
# =========================================================
# 이 노트북은 Google Colab에서 단독 실행되도록 구성되어 있습니다.
# 다른 사용자는 아래 ROOT_PATH만 본인의 Google Drive 경로로 수정하면 됩니다.

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "IPython": "ipython",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(f"[설치] 필요한 패키지 설치: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("[확인] 필요한 패키지가 이미 설치되어 있습니다.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("[확인] Google Drive mount 완료")

    # 그래프 한글 표시용 폰트입니다. 설치 실패 시에도 계산은 계속 진행됩니다.
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum", "-qq"], check=False)
except ModuleNotFoundError:
    print("[안내] Colab 환경이 아니므로 Google Drive mount와 apt 폰트 설치를 건너뜁니다.")

# 현재 작업자가 사용한 원본 경로입니다.
# 다른 사용자는 이 경로만 본인 Drive 구조에 맞게 수정하면 됩니다.
ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
DATA_PATH = ROOT_PATH + "data/"
PROCESSED_PATH = ROOT_PATH + "preprocessed_data/"

print(f"[경로] ROOT_PATH      = {ROOT_PATH}")
print(f"[경로] DATA_PATH      = {DATA_PATH}")
print(f"[경로] PROCESSED_PATH = {PROCESSED_PATH}")
print(f"[경로] 결과 폴더       = {ROOT_PATH}정책적용_v2/")


## 3단계: 국내 정책 적용

### 함수정의

In [ ]:
# =========================================================
# 3단계-v2: 국내 정책 적용
# =========================================================
# 목적:
# 1) 2단계-v2 결과(pred_gross = 정책 미반영 적정가격)를 읽는다.
# 2) 카테고리 0 유류세 인하를 주유소 적정가격/범위에 수치 반영한다.
# 3) 카테고리 1 최고가격제는 주유소 가격에 직접 차감하지 않고,
#    정유사 weekly 결과에서 별도 감사표로 점검한다.
# 4) 결과는 ROOT_PATH/정책적용_v2/ 아래에 저장한다.
# =========================================================

from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm


# =========================================================
# 0) 기본 설정
# =========================================================
plt.rcdefaults()

available_fonts = {f.name for f in fm.fontManager.ttflist}
if "NanumGothic" in available_fonts:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",

    "axes.edgecolor": "black",
    "axes.linewidth": 1.0,

    "xtick.color": "black",
    "ytick.color": "black",

    "grid.color": "#d9d9d9",
    "grid.linestyle": "-",
    "grid.linewidth": 0.8,
    "grid.alpha": 0.6,
})


def _ensure_trailing_slash(x):
    x = str(x)
    return x if x.endswith("/") else x + "/"


def _resolve_path(primary_path, fallback_root=None):
    primary_path = Path(primary_path)

    if primary_path.exists():
        return str(primary_path)

    if fallback_root is not None:
        alt = Path(fallback_root) / primary_path.name
        if alt.exists():
            print(f"[WARN] 기본 경로가 없어 fallback 경로 사용: {alt}")
            return str(alt)

    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {primary_path}")


def _resolve_integrated_path(processed_path, fallback_root=None):
    base = Path(processed_path)

    candidates = [
        base / "분석용_일별_통합데이터.csv",
        base / "분석용_일별_통합데이터.csv",
    ]

    for c in candidates:
        if c.exists():
            return str(c)

    matches = sorted(base.glob("*통합데이터*.csv"))
    if matches:
        return str(matches[0])

    if fallback_root is not None:
        fb = Path(fallback_root)
        matches = sorted(fb.glob("*통합데이터*.csv"))
        if matches:
            print(f"[WARN] fallback 통합데이터 사용: {matches[0]}")
            return str(matches[0])

    raise FileNotFoundError(f"통합데이터 파일을 찾지 못했습니다: {processed_path}")


# =========================================================
# 1) 경로 설정
# =========================================================
ROOT_PATH = _ensure_trailing_slash(ROOT_PATH)
DATA_PATH = _ensure_trailing_slash(DATA_PATH)
PROCESSED_PATH = _ensure_trailing_slash(PROCESSED_PATH)

STEP2_OUTPUT_DIR = Path(ROOT_PATH) / "적정가격대선정_v2"

GASOLINE_PRED_PATH = STEP2_OUTPUT_DIR / "gasoline_production_predictions_full_calendar.csv"
DIESEL_PRED_PATH = STEP2_OUTPUT_DIR / "diesel_production_predictions_full_calendar.csv"

GASOLINE_REFINERY_PRED_PATH = STEP2_OUTPUT_DIR / "gasoline" / "gasoline_refinery_weekly_predictions.csv"
DIESEL_REFINERY_PRED_PATH = STEP2_OUTPUT_DIR / "diesel" / "diesel_refinery_weekly_predictions.csv"

POLICY_PATH = Path(DATA_PATH) / "korea_fuel_tax_price_policies.csv"
INTEGRATED_PATH = _resolve_integrated_path(PROCESSED_PATH, ROOT_PATH)

GASOLINE_PRED_PATH = _resolve_path(GASOLINE_PRED_PATH, ROOT_PATH)
DIESEL_PRED_PATH = _resolve_path(DIESEL_PRED_PATH, ROOT_PATH)
POLICY_PATH = _resolve_path(POLICY_PATH, ROOT_PATH)
INTEGRATED_PATH = _resolve_path(INTEGRATED_PATH, ROOT_PATH)

OUTPUT_BASE = Path(ROOT_PATH) / "정책적용_v2"
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)


# =========================================================
# 2) 정책 적용 옵션
# =========================================================
# 2단계-v2 결과는 pred_gross/band_low/band_high가 정책 미반영 기준이다.
BASELINE_INPUT_MODE = "prediction_is_already_nonpolicy"

# 소비자가격 기준 정책효과는 VAT 포함 기준을 기본값으로 둔다.
# 필요하면 "official_tax_cut_exvat"로 바꿀 수 있다.
FAIR_SHIFT_MODE = "official_tax_cut_vat_inclusive"

# 정책 인하분이 주유소 소비자가격에 전액 반영된다는 기준.
# 실제 전가율 시나리오를 보고 싶으면 0.5, 0.7 등으로 바꿔 별도 실행.
RETAIL_PASS_THROUGH_RATE = 1.0

# 법적 효력일 기준. 지연 반영 실험 시 7, 14 등으로 변경.
RETAIL_PASS_THROUGH_LAG_DAYS = 0

# 카테고리 0 = 유류세 인하율, 카테고리 1 = 정유사 최고가격제
NUMERICALLY_APPLY_POLICY_CATEGORIES = [0]

# 기준 유류세, ex-VAT, 원/L
BASE_FUEL_TAX_EXVAT_원L = {
    "휘발유": 745.89,
    "경유": 528.75,
}

VAT_RATE = 0.10


# =========================================================
# 3) 그래프 색상
# =========================================================
COLOR_ACTUAL = "#1f77b4"

# 기간별 적정범위 색상
COLOR_BAND_NONE = "lightblue"     # 아무것도 없는 기간: 기존 2단계 색상
COLOR_BAND_TAX = "mistyrose"      # 유류세 인하 기간
COLOR_BAND_CAP = "#fff2cc"        # 최고가격제 기간

# 적정가격 선 색상
COLOR_FAIR_NOPOLICY = "#ff7f0e"   # 이전/미정책 적정가격: 기존 주황색
COLOR_FAIR_POLICY = "#9467bd"     # 정책 적용 적정가격: 보라색

# 실제 가격 및 이탈점
COLOR_ABOVE = "#d62728"
COLOR_BELOW = "#2ca02c"

# =========================================================
# 4) 데이터 로드
# =========================================================
gas_pred = pd.read_csv(GASOLINE_PRED_PATH, parse_dates=["date"])
diesel_pred = pd.read_csv(DIESEL_PRED_PATH, parse_dates=["date"])

policy = pd.read_csv(POLICY_PATH)
integrated = pd.read_csv(INTEGRATED_PATH, parse_dates=["date"])

policy["시작일"] = pd.to_datetime(policy["시작일"], errors="coerce")
policy["종료일"] = pd.to_datetime(policy["종료일"], errors="coerce")
policy["가격"] = pd.to_numeric(policy["가격"], errors="coerce")
policy["카테고리"] = pd.to_numeric(policy["카테고리"], errors="coerce").astype("Int64")

policy = (
    policy
    .dropna(subset=["시작일", "종료일", "유종", "카테고리"])
    .sort_values(["유종", "시작일", "종료일", "카테고리"])
    .reset_index(drop=True)
)

DATA_MIN = min(gas_pred["date"].min(), diesel_pred["date"].min(), integrated["date"].min())
DATA_MAX = max(gas_pred["date"].max(), diesel_pred["date"].max(), integrated["date"].max())

print(f"[DATA RANGE] {DATA_MIN.date()} ~ {DATA_MAX.date()}")
print(f"[GAS PRED] {GASOLINE_PRED_PATH}")
print(f"[DIESEL PRED] {DIESEL_PRED_PATH}")
print(f"[POLICY] {POLICY_PATH}")
print(f"[INTEGRATED] {INTEGRATED_PATH}")
print(f"[OUTPUT_BASE] {OUTPUT_BASE}")


# =========================================================
# 5) prediction 파일 구조 검증
# =========================================================
def validate_prediction_file_v2(df: pd.DataFrame, fuel_name: str) -> pd.DataFrame:
    rows = []

    required = ["date", "pred_gross", "band_low", "band_high"]
    missing = [c for c in required if c not in df.columns]

    rows.append({
        "유종": fuel_name,
        "검증항목": "필수컬럼",
        "상태": "OK" if not missing else "FAIL",
        "값": "" if not missing else ",".join(missing),
        "설명": "2단계-v2 결과는 date, pred_gross, band_low, band_high가 필요",
    })

    if missing:
        return pd.DataFrame(rows)

    for col in ["pred_gross", "band_low", "band_high"]:
        rows.append({
            "유종": fuel_name,
            "검증항목": f"{col}_non_null",
            "상태": "OK",
            "값": int(pd.to_numeric(df[col], errors="coerce").notna().sum()),
            "설명": f"{col} 비결측 개수",
        })

    # 구버전 2단계 파일 오투입 감지
    if all(c in df.columns for c in ["pred_pretax", "tax_sum_full"]):
        chk = df.loc[
            df[["pred_gross", "pred_pretax", "tax_sum_full"]].notna().all(axis=1)
        ].copy()

        if len(chk) > 0:
            max_abs_diff = (
                chk["pred_gross"] - (chk["pred_pretax"] + chk["tax_sum_full"])
            ).abs().max()

            rows.append({
                "유종": fuel_name,
                "검증항목": "old_structure_detection",
                "상태": "WARN" if max_abs_diff < 1e-6 else "OK",
                "값": max_abs_diff,
                "설명": (
                    "0에 가까우면 구버전 파일일 가능성. "
                    "3단계-v2는 pred_gross를 정책 미반영 적정가격으로 해석함."
                ),
            })
        else:
            rows.append({
                "유종": fuel_name,
                "검증항목": "old_structure_detection",
                "상태": "OK",
                "값": "pred_pretax/tax_sum_full usable rows 없음",
                "설명": "2단계-v2 placeholder 구조로 판단 가능",
            })
    else:
        rows.append({
            "유종": fuel_name,
            "검증항목": "old_structure_detection",
            "상태": "OK",
            "값": "old columns absent",
            "설명": "tax_sum_full에 의존하지 않는 v2 구조",
        })

    # band 순서 검증
    bad_band = (
        pd.to_numeric(df["band_low"], errors="coerce")
        > pd.to_numeric(df["band_high"], errors="coerce")
    ).sum()

    rows.append({
        "유종": fuel_name,
        "검증항목": "band_low_le_band_high",
        "상태": "OK" if bad_band == 0 else "FAIL",
        "값": int(bad_band),
        "설명": "band_low > band_high 행 수",
    })

    return pd.DataFrame(rows)


validation_df = pd.concat([
    validate_prediction_file_v2(gas_pred, "휘발유"),
    validate_prediction_file_v2(diesel_pred, "경유"),
], ignore_index=True)

validation_df.to_csv(
    OUTPUT_BASE / "prediction_구조검증_v2.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n[prediction 구조 검증]")
print(validation_df.to_string(index=False))


# =========================================================
# 6) 정책 스케줄 일별화
# =========================================================
def build_daily_policy(date_index_df: pd.DataFrame,
                       policy_df: pd.DataFrame,
                       fuel_name: str,
                       lag_days: int = 0) -> pd.DataFrame:
    out = (
        date_index_df[["date"]]
        .drop_duplicates()
        .sort_values("date")
        .reset_index(drop=True)
        .copy()
    )

    out["유종"] = fuel_name

    # 카테고리 0: 유류세 인하
    out["유류세정책명"] = pd.NA
    out["유류세인하율_pct"] = 0.0
    out["유류세정책_적용여부"] = False

    # 카테고리 1: 정유사 최고가격제
    out["최고가격제정책명"] = pd.NA
    out["정유사공급가격상한_원L"] = np.nan
    out["최고가격제_적용여부"] = False

    fuel_policy = (
        policy_df[policy_df["유종"] == fuel_name]
        .copy()
        .sort_values(["시작일", "종료일", "카테고리"])
    )

    for _, row in fuel_policy.iterrows():
        start = row["시작일"] + pd.Timedelta(days=lag_days)
        end = row["종료일"] + pd.Timedelta(days=lag_days)
        mask = (out["date"] >= start) & (out["date"] <= end)

        if int(row["카테고리"]) == 0:
            out.loc[mask, "유류세정책명"] = row["정책명"]
            out.loc[mask, "유류세인하율_pct"] = abs(float(row["가격"]))
            out.loc[mask, "유류세정책_적용여부"] = True

        elif int(row["카테고리"]) == 1:
            out.loc[mask, "최고가격제정책명"] = row["정책명"]
            out.loc[mask, "정유사공급가격상한_원L"] = float(row["가격"])
            out.loc[mask, "최고가격제_적용여부"] = True

    out["정책적용여부_전체"] = out["유류세정책_적용여부"] | out["최고가격제_적용여부"]

    # ---------------------------------------------------------
    # 그래프 구간 구분
    # ---------------------------------------------------------
    # 최종 주유소 가격에 수치 반영되는 정책은 유류세 인하뿐이다.
    # 다만 최고가격제 기간은 주유소 가격에 직접 차감하지 않더라도
    # 별도 정책 기간으로 색상 표시한다.
    #
    # 겹치는 경우:
    # 유류세 인하와 최고가격제가 동시에 적용되면
    # 그래프 색상은 최고가격제 기간으로 우선 표시한다.
    # 가격 계산은 유류세 인하가 있으면 그대로 반영된다.
    # ---------------------------------------------------------
    out["그래프_구간"] = "아무것도 없는 기간"

    out.loc[
        out["유류세정책_적용여부"].fillna(False),
        "그래프_구간"
    ] = "유류세 인하 기간"

    out.loc[
        out["최고가격제_적용여부"].fillna(False),
        "그래프_구간"
    ] = "최고가격제 기간"

    # 기존 코드 호환용
    out["그래프_핑크기간"] = out["유류세정책_적용여부"]

    def make_label(r):
        labels = []

        if pd.notna(r["유류세정책명"]):
            labels.append(f'{r["유류세정책명"]} ({float(r["유류세인하율_pct"]):g}%)')

        if pd.notna(r["최고가격제정책명"]):
            if pd.notna(r["정유사공급가격상한_원L"]):
                labels.append(
                    f'{r["최고가격제정책명"]} '
                    f'({int(r["정유사공급가격상한_원L"])}원/L, 정유사 단계 감사)'
                )
            else:
                labels.append(f'{r["최고가격제정책명"]} (정유사 단계 감사)')

        return " | ".join(labels) if labels else ""

    out["정책라벨"] = out.apply(make_label, axis=1)

    return out


# =========================================================
# 7) 유류세 감면액 계산
# =========================================================
def compute_policy_shift(policy_daily: pd.DataFrame, fuel_name: str) -> pd.DataFrame:
    out = policy_daily.copy()

    base_tax_exvat = BASE_FUEL_TAX_EXVAT_원L[fuel_name]

    rate = pd.to_numeric(out["유류세인하율_pct"], errors="coerce").fillna(0.0) / 100.0
    valid_rate = (rate > 0) & (rate < 1)

    out["기준유류세_exVAT_원L"] = base_tax_exvat

    out["정책감면액_exVAT_원L"] = 0.0
    out.loc[valid_rate, "정책감면액_exVAT_원L"] = base_tax_exvat * rate.loc[valid_rate]

    out["정책감면액_VAT포함참고_원L"] = out["정책감면액_exVAT_원L"] * (1.0 + VAT_RATE)

    if FAIR_SHIFT_MODE == "official_tax_cut_vat_inclusive":
        out["정책적용_시프트_원L"] = out["정책감면액_VAT포함참고_원L"]

    elif FAIR_SHIFT_MODE == "official_tax_cut_exvat":
        out["정책적용_시프트_원L"] = out["정책감면액_exVAT_원L"]

    else:
        raise ValueError(
            "FAIR_SHIFT_MODE는 'official_tax_cut_vat_inclusive' 또는 "
            "'official_tax_cut_exvat' 여야 합니다."
        )

    out["정책적용_시프트_원L"] = out["정책적용_시프트_원L"] * float(RETAIL_PASS_THROUGH_RATE)

    return out


# =========================================================
# 8) 범위 판정 helper
# =========================================================
def classify_position(actual: pd.Series, low: pd.Series, high: pd.Series):
    valid = actual.notna() & low.notna() & high.notna()
    inside = valid & (actual >= low) & (actual <= high)
    above = valid & (actual > high)
    below = valid & (actual < low)
    return valid, inside, above, below


# =========================================================
# 9) 유종별 일별 데이터 생성
# =========================================================
def build_fuel_daily(pred_df: pd.DataFrame,
                     actual_col: str,
                     fuel_name: str) -> pd.DataFrame:
    pred_df = pred_df.copy().sort_values("date").reset_index(drop=True)

    actual_df = (
        integrated[["date", actual_col]]
        .copy()
        .rename(columns={actual_col: "국내유가_원L"})
    )

    policy_daily = build_daily_policy(
        pred_df[["date"]],
        policy,
        fuel_name,
        lag_days=RETAIL_PASS_THROUGH_LAG_DAYS,
    )
    policy_daily = compute_policy_shift(policy_daily, fuel_name)

    df = pred_df.merge(actual_df, on="date", how="left")
    df = df.merge(policy_daily, on="date", how="left")

    if BASELINE_INPUT_MODE != "prediction_is_already_nonpolicy":
        raise ValueError("3단계-v2에서는 BASELINE_INPUT_MODE='prediction_is_already_nonpolicy'만 사용하세요.")

    required = ["pred_gross", "band_low", "band_high"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{fuel_name} prediction 파일 필수 컬럼 누락: {missing}")

    shift = pd.to_numeric(df["정책적용_시프트_원L"], errors="coerce").fillna(0.0)

    # 2단계-v2 결과는 정책 미반영 기준
    df["적정가격_미정책_원L"] = df["pred_gross"]
    df["적정범위_미정책_하한_원L"] = df["band_low"]
    df["적정범위_미정책_상한_원L"] = df["band_high"]

    # 카테고리 0 유류세 인하 수치 반영
    df["적정가격_정책적용_원L"] = df["적정가격_미정책_원L"] - shift
    df["적정범위_정책적용_하한_원L"] = df["적정범위_미정책_하한_원L"] - shift
    df["적정범위_정책적용_상한_원L"] = df["적정범위_미정책_상한_원L"] - shift

    df["정책효과_원L"] = df["적정가격_미정책_원L"] - df["적정가격_정책적용_원L"]
    df["미정책_vs_pred_gross_diff_원L"] = df["적정가격_미정책_원L"] - df["pred_gross"]

    # 범위 판정
    valid0, inside0, above0, below0 = classify_position(
        df["국내유가_원L"],
        df["적정범위_미정책_하한_원L"],
        df["적정범위_미정책_상한_원L"],
    )

    valid1, inside1, above1, below1 = classify_position(
        df["국내유가_원L"],
        df["적정범위_정책적용_하한_원L"],
        df["적정범위_정책적용_상한_원L"],
    )

    df["미정책_valid"] = valid0
    df["미정책_inside"] = inside0
    df["미정책_above"] = above0
    df["미정책_below"] = below0

    df["정책적용_valid"] = valid1
    df["정책적용_inside"] = inside1
    df["정책적용_above"] = above1
    df["정책적용_below"] = below1

    df["최고가격제_수치반영여부"] = np.where(
        df["최고가격제_적용여부"].fillna(False),
        "주유소 가격에는 직접 수치 미반영. 정유사 weekly 감사표에서 별도 점검",
        "",
    )

    df["유종"] = fuel_name

    ordered_cols = [
        "date", "유종", "국내유가_원L",

        "적정가격_미정책_원L",
        "적정범위_미정책_하한_원L",
        "적정범위_미정책_상한_원L",

        "적정가격_정책적용_원L",
        "적정범위_정책적용_하한_원L",
        "적정범위_정책적용_상한_원L",

        "정책효과_원L",
        "미정책_vs_pred_gross_diff_원L",

        "pred_gross", "band_low", "band_high",

        "기준유류세_exVAT_원L",
        "정책감면액_exVAT_원L",
        "정책감면액_VAT포함참고_원L",
        "정책적용_시프트_원L",

        "유류세정책명", "유류세인하율_pct", "유류세정책_적용여부",
        "최고가격제정책명", "정유사공급가격상한_원L", "최고가격제_적용여부",
        "최고가격제_수치반영여부",
        "정책적용여부_전체", "그래프_핑크기간", "정책라벨",

        "selected_model_name", "selected_band_coverage",
        "fair_retail_direct", "fair_retail_two_layer",
        "fair_refinery_pre", "pred_retail_spread",
        "retail_spread_vs_fair_refinery", "retail_gap_원L",

        "미정책_valid", "미정책_inside", "미정책_above", "미정책_below",
        "정책적용_valid", "정책적용_inside", "정책적용_above", "정책적용_below",
    ]

    existing_cols = [c for c in ordered_cols if c in df.columns]
    df = df[existing_cols].sort_values("date").reset_index(drop=True)

    return df


# =========================================================
# 10) 그래프 helper
# =========================================================
def dedup_legend(ax):
    handles, labels = ax.get_legend_handles_labels()
    unique = {}

    for h, l in zip(handles, labels):
        if l not in unique:
            unique[l] = h

    ax.legend(list(unique.values()), list(unique.keys()), loc="best")


def fill_band_segments(ax, df, mask, low_col, high_col, color, alpha, label):
    seg = df.loc[mask, ["date", low_col, high_col]].dropna().copy()

    if seg.empty:
        return

    seg = seg.sort_values("date").reset_index(drop=True)
    seg["gap"] = seg["date"].diff().dt.days.fillna(1).ne(1)
    seg["grp"] = seg["gap"].cumsum()

    first = True

    for _, part in seg.groupby("grp"):
        ax.fill_between(
            part["date"],
            part[low_col],
            part[high_col],
            color=color,
            alpha=alpha,
            label=label if first else None,
        )
        first = False


def plot_line_segments(ax, df, mask, y_col, label, **kwargs):
    seg = df.loc[mask, ["date", y_col]].dropna().copy()

    if seg.empty:
        return

    seg = seg.sort_values("date").reset_index(drop=True)
    seg["gap"] = seg["date"].diff().dt.days.fillna(1).ne(1)
    seg["grp"] = seg["gap"].cumsum()

    first = True

    for _, part in seg.groupby("grp"):
        ax.plot(
            part["date"],
            part[y_col],
            label=label if first else None,
            **kwargs,
        )
        first = False


def apply_clean_axis_style(ax):
    ax.set_facecolor("white")

    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("black")
        ax.spines[side].set_linewidth(1.0)

    ax.tick_params(axis="both", colors="black", width=1.0)
    ax.grid(True, color="#d9d9d9", linestyle="-", linewidth=0.8, alpha=0.6)


# =========================================================
# 11) 연도별 그래프
# =========================================================
def plot_yearly_policy_applied(df: pd.DataFrame,
                               fuel_name: str,
                               save_dir: Path):
    save_dir.mkdir(parents=True, exist_ok=True)

    years = sorted(df["date"].dt.year.dropna().unique())

    for year in years:
        d = (
            df[df["date"].dt.year == year]
            .copy()
            .sort_values("date")
            .reset_index(drop=True)
        )

        if d.empty:
            continue

        # 그래프_구간 컬럼이 없는 구버전 daily_df 방어
        if "그래프_구간" not in d.columns:
            d["그래프_구간"] = "아무것도 없는 기간"
            d.loc[
                d["유류세정책_적용여부"].fillna(False),
                "그래프_구간"
            ] = "유류세 인하 기간"
            d.loc[
                d["최고가격제_적용여부"].fillna(False),
                "그래프_구간"
            ] = "최고가격제 기간"

        fig, ax = plt.subplots(figsize=(20, 8))
        fig.patch.set_facecolor("white")
        ax.set_facecolor("white")

        # -------------------------------------------------
        # 적정범위 유효성
        # -------------------------------------------------
        policy_band_valid = (
            d["적정범위_정책적용_하한_원L"].notna()
            & d["적정범위_정책적용_상한_원L"].notna()
        )

        none_mask = (
            d["그래프_구간"].eq("아무것도 없는 기간")
            & policy_band_valid
        )

        tax_mask = (
            d["그래프_구간"].eq("유류세 인하 기간")
            & policy_band_valid
        )

        cap_mask = (
            d["그래프_구간"].eq("최고가격제 기간")
            & policy_band_valid
        )

        # -------------------------------------------------
        # 1) 아무것도 없는 기간: 2단계 적정범위 색상
        # -------------------------------------------------
        fill_band_segments(
            ax=ax,
            df=d,
            mask=none_mask,
            low_col="적정범위_정책적용_하한_원L",
            high_col="적정범위_정책적용_상한_원L",
            color=COLOR_BAND_NONE,
            alpha=0.45,
            label="적정 범위(무정책 기간)",
        )

        # -------------------------------------------------
        # 2) 유류세 인하 기간: 유류세 반영 적정범위
        # -------------------------------------------------
        fill_band_segments(
            ax=ax,
            df=d,
            mask=tax_mask,
            low_col="적정범위_정책적용_하한_원L",
            high_col="적정범위_정책적용_상한_원L",
            color=COLOR_BAND_TAX,
            alpha=0.65,
            label="적정 범위(유류세 인하 반영)",
        )

        # -------------------------------------------------
        # 3) 최고가격제 기간: 별도 색상
        # 주유소 가격에 직접 차감하지는 않음.
        # 다만 유류세 인하와 겹치는 경우, 가격대는 유류세 반영값이고
        # 색상은 최고가격제 기간으로 표시된다.
        # -------------------------------------------------
        fill_band_segments(
            ax=ax,
            df=d,
            mask=cap_mask,
            low_col="적정범위_정책적용_하한_원L",
            high_col="적정범위_정책적용_상한_원L",
            color=COLOR_BAND_CAP,
            alpha=0.75,
            label="적정 범위(최고가격제 기간)",
        )

        # -------------------------------------------------
        # 실제 국내 유가
        # -------------------------------------------------
        ax.plot(
            d["date"],
            d["국내유가_원L"],
            color=COLOR_ACTUAL,
            linewidth=1.8,
            label="국내 유가",
        )

        # -------------------------------------------------
        # 이전/미정책 적정가격
        # 전체 기간 표시
        # -------------------------------------------------
        ax.plot(
            d["date"],
            d["적정가격_미정책_원L"],
            color=COLOR_FAIR_NOPOLICY,
            linewidth=1.8,
            label="이전 적정 가격",
        )

        # -------------------------------------------------
        # 정책 적용 적정가격
        # 유류세 인하가 실제로 수치 반영되는 기간에만 표시
        # -------------------------------------------------
        policy_price_mask = (
            d["유류세정책_적용여부"].fillna(False)
            & d["적정가격_정책적용_원L"].notna()
        )

        plot_line_segments(
            ax=ax,
            df=d,
            mask=policy_price_mask,
            y_col="적정가격_정책적용_원L",
            label="정책 적용 적정 가격",
            color=COLOR_FAIR_POLICY,
            linewidth=2.0,
            linestyle="-",
        )

        # -------------------------------------------------
        # 정책 기간의 이전 적정범위는 면으로 칠하지 않고 점선만 표시
        # 이렇게 해야 이전 범위와 적용 범위를 동시에 보면서도
        # 둘 사이가 넓은 범위처럼 오해되지 않는다.
        # -------------------------------------------------
        any_policy_mask = (
            d["정책적용여부_전체"].fillna(False)
            & d["적정범위_미정책_하한_원L"].notna()
            & d["적정범위_미정책_상한_원L"].notna()
        )

        plot_line_segments(
            ax=ax,
            df=d,
            mask=any_policy_mask,
            y_col="적정범위_미정책_하한_원L",
            label="이전 적정 범위 하한",
            color=COLOR_FAIR_NOPOLICY,
            linewidth=1.0,
            linestyle=":",
        )

        plot_line_segments(
            ax=ax,
            df=d,
            mask=any_policy_mask,
            y_col="적정범위_미정책_상한_원L",
            label="이전 적정 범위 상한",
            color=COLOR_FAIR_NOPOLICY,
            linewidth=1.0,
            linestyle=":",
        )

        # -------------------------------------------------
        # 상향 / 하향 이탈
        # 판정 기준은 정책 적용 적정범위
        # -------------------------------------------------
        above = d["정책적용_above"].fillna(False)
        below = d["정책적용_below"].fillna(False)

        if above.any():
            ax.scatter(
                d.loc[above, "date"],
                d.loc[above, "국내유가_원L"],
                color=COLOR_ABOVE,
                s=16,
                label="상향 이탈",
                zorder=3,
            )

        if below.any():
            ax.scatter(
                d.loc[below, "date"],
                d.loc[below, "국내유가_원L"],
                color=COLOR_BELOW,
                s=16,
                label="하향 이탈",
                zorder=3,
            )

        # -------------------------------------------------
        # 제목 / 축
        # -------------------------------------------------
        title = f"{year}년 {fuel_name} 실제 유가 vs 이전/정책 적용 적정 가격·범위"

        if d["최고가격제_적용여부"].any():
            title += " (최고가격제 기간 포함, 주유소 직접 차감 없음)"

        ax.set_title(title)
        ax.set_xlabel("date")
        ax.set_ylabel("원/리터")

        if len(d) <= 130:
            ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        else:
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
        apply_clean_axis_style(ax)
        dedup_legend(ax)

        out_path = save_dir / f"{year}_{fuel_name}_policy_applied_fair_band.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        print(f"[SAVE] {out_path}")


# =========================================================
# 12) 요약 함수
# =========================================================
def summarize_counts(df: pd.DataFrame, fuel_name: str) -> pd.DataFrame:
    def pack(label: str, prefix: str) -> dict:
        return {
            "유종": fuel_name,
            "구분": label,
            "판정대상일수": int(df[f"{prefix}_valid"].sum()),
            "범위내": int(df[f"{prefix}_inside"].sum()),
            "상향이탈": int(df[f"{prefix}_above"].sum()),
            "하향이탈": int(df[f"{prefix}_below"].sum()),
        }

    s0 = pack("미정책", "미정책")
    s1 = pack("정책적용", "정책적용")

    delta = {
        "유종": fuel_name,
        "구분": "변화량(정책적용-미정책)",
        "판정대상일수": s1["판정대상일수"] - s0["판정대상일수"],
        "범위내": s1["범위내"] - s0["범위내"],
        "상향이탈": s1["상향이탈"] - s0["상향이탈"],
        "하향이탈": s1["하향이탈"] - s0["하향이탈"],
    }

    return pd.DataFrame([s0, s1, delta])


def summarize_counts_by_year(df: pd.DataFrame, fuel_name: str) -> pd.DataFrame:
    rows = []

    for year, g in df.groupby(df["date"].dt.year):
        rows.append({
            "유종": fuel_name,
            "연도": int(year),
            "구분": "미정책",
            "판정대상일수": int(g["미정책_valid"].sum()),
            "범위내": int(g["미정책_inside"].sum()),
            "상향이탈": int(g["미정책_above"].sum()),
            "하향이탈": int(g["미정책_below"].sum()),
        })

        rows.append({
            "유종": fuel_name,
            "연도": int(year),
            "구분": "정책적용",
            "판정대상일수": int(g["정책적용_valid"].sum()),
            "범위내": int(g["정책적용_inside"].sum()),
            "상향이탈": int(g["정책적용_above"].sum()),
            "하향이탈": int(g["정책적용_below"].sum()),
        })

        rows.append({
            "유종": fuel_name,
            "연도": int(year),
            "구분": "변화량(정책적용-미정책)",
            "판정대상일수": int(g["정책적용_valid"].sum() - g["미정책_valid"].sum()),
            "범위내": int(g["정책적용_inside"].sum() - g["미정책_inside"].sum()),
            "상향이탈": int(g["정책적용_above"].sum() - g["미정책_above"].sum()),
            "하향이탈": int(g["정책적용_below"].sum() - g["미정책_below"].sum()),
        })

    return pd.DataFrame(rows)


def policy_effect_summary(df: pd.DataFrame, fuel_name: str) -> pd.DataFrame:
    z = df[df["유류세정책_적용여부"].fillna(False)].copy()

    if z.empty:
        return pd.DataFrame([{
            "유종": fuel_name,
            "유류세정책명": "정책기간 없음",
            "유류세인하율_pct": np.nan,
            "시작일": pd.NaT,
            "종료일": pd.NaT,
            "일수": 0,
            "평균정책효과_원L": np.nan,
            "최소정책효과_원L": np.nan,
            "최대정책효과_원L": np.nan,
        }])

    return (
        z.groupby(["유종", "유류세정책명", "유류세인하율_pct"], dropna=False)
        .agg(
            시작일=("date", "min"),
            종료일=("date", "max"),
            일수=("date", "count"),
            평균정책효과_원L=("정책효과_원L", "mean"),
            최소정책효과_원L=("정책효과_원L", "min"),
            최대정책효과_원L=("정책효과_원L", "max"),
            평균감면액_exVAT_원L=("정책감면액_exVAT_원L", "mean"),
            평균감면액_VAT포함참고_원L=("정책감면액_VAT포함참고_원L", "mean"),
        )
        .reset_index()
    )


def build_policy_shift_audit(daily_df: pd.DataFrame, fuel_name: str) -> pd.DataFrame:
    fuel_policy = (
        policy[(policy["유종"] == fuel_name) & (policy["카테고리"] == 0)]
        .copy()
        .sort_values("시작일")
    )

    rows = []

    for _, row in fuel_policy.iterrows():
        start_dt = row["시작일"] + pd.Timedelta(days=RETAIL_PASS_THROUGH_LAG_DAYS)

        z = daily_df[daily_df["date"] == start_dt].copy()
        if z.empty:
            continue

        r = z.iloc[0]

        rows.append({
            "유종": fuel_name,
            "정책시작일": start_dt.date(),
            "정책명": row["정책명"],
            "인하율_pct": abs(float(row["가격"])),
            "정책효과_원L": r.get("정책효과_원L", np.nan),
            "정책감면액_exVAT_원L": r.get("정책감면액_exVAT_원L", np.nan),
            "정책감면액_VAT포함참고_원L": r.get("정책감면액_VAT포함참고_원L", np.nan),
            "미정책가격": r.get("적정가격_미정책_원L", np.nan),
            "정책적용가격": r.get("적정가격_정책적용_원L", np.nan),
        })

    return pd.DataFrame(rows)


def build_policy_applicability_report(policy_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in policy_df.iterrows():
        overlap_start = max(row["시작일"], DATA_MIN)
        overlap_end = min(row["종료일"], DATA_MAX)
        overlap_days = max((overlap_end - overlap_start).days + 1, 0)

        if overlap_days == 0:
            status = "데이터구간밖"
            reason = "업로드 데이터 기간과 겹치지 않음"

        else:
            if int(row["카테고리"]) in NUMERICALLY_APPLY_POLICY_CATEGORIES:
                status = "수치반영"
                reason = "유류세율(카테고리 0)은 주유소 적정가격에서 차감"
            else:
                status = "정유사단계감사"
                reason = "정유사 공급가격 상한은 정유사 weekly 결과에서 별도 점검"

        rows.append({
            "유종": row["유종"],
            "정책명": row["정책명"],
            "카테고리": int(row["카테고리"]),
            "가격": row["가격"],
            "시작일": row["시작일"].date(),
            "종료일": row["종료일"].date(),
            "데이터겹침일수": overlap_days,
            "처리결과": status,
            "사유": reason,
        })

    return pd.DataFrame(rows)


def audit_policy_snapshot(snapshot_dates=None) -> pd.DataFrame:
    if snapshot_dates is None:
        snapshot_dates = [
            "2021-11-11",
            "2021-11-12",
            "2022-07-01",
            "2025-11-01",
            "2026-03-26",
            "2026-03-27",
        ]

    dates = pd.DataFrame({"date": pd.to_datetime(snapshot_dates)})
    rows = []

    for fuel_name in ["휘발유", "경유"]:
        p = build_daily_policy(dates, policy, fuel_name, lag_days=RETAIL_PASS_THROUGH_LAG_DAYS)
        p = compute_policy_shift(p, fuel_name)

        for _, r in p.iterrows():
            rows.append({
                "date": r["date"],
                "유종": fuel_name,
                "유류세정책명": r["유류세정책명"],
                "유류세인하율_pct": r["유류세인하율_pct"],
                "정책감면액_exVAT_원L": r["정책감면액_exVAT_원L"],
                "정책감면액_VAT포함참고_원L": r["정책감면액_VAT포함참고_원L"],
                "정책적용_시프트_원L": r["정책적용_시프트_원L"],
                "정책라벨": r["정책라벨"],
            })

    return pd.DataFrame(rows)

# =========================================================
# 12-B) 전체 / 정책기간 기준 범위판정 요약
# =========================================================
def summarize_range_counts_by_scope(
    df: pd.DataFrame,
    fuel_name: str,
    scope_name: str,
    mask: pd.Series,
) -> pd.DataFrame:
    g = df.loc[mask.fillna(False)].copy()

    rows = []

    for 기준명, prefix in [
        ("이전 적정 범위 기준", "미정책"),
        ("적용 적정 범위 기준", "정책적용"),
    ]:
        rows.append({
            "유종": fuel_name,
            "구간": scope_name,
            "기준": 기준명,
            "판정대상일수": int(g[f"{prefix}_valid"].fillna(False).sum()),
            "적정": int(g[f"{prefix}_inside"].fillna(False).sum()),
            "상향": int(g[f"{prefix}_above"].fillna(False).sum()),
            "하향": int(g[f"{prefix}_below"].fillna(False).sum()),
        })

    return pd.DataFrame(rows)


def build_range_count_report(df: pd.DataFrame, fuel_name: str) -> pd.DataFrame:
    all_mask = pd.Series(True, index=df.index)

    # 주유소 가격에 실제 수치 반영한 정책은 유류세 인하뿐이다.
    # 최고가격제는 주유소 가격에 직접 반영하지 않으므로 여기서 제외한다.
    policy_mask = df["유류세정책_적용여부"].fillna(False)

    out = pd.concat(
        [
            summarize_range_counts_by_scope(
                df=df,
                fuel_name=fuel_name,
                scope_name="전체 기준",
                mask=all_mask,
            ),
            summarize_range_counts_by_scope(
                df=df,
                fuel_name=fuel_name,
                scope_name="유류세 정책 적용 기간 기준",
                mask=policy_mask,
            ),
        ],
        ignore_index=True,
    )

    return out

# =========================================================
# 13) 정유사 최고가격제 weekly 감사
# =========================================================
def load_optional_refinery_pred(path) -> pd.DataFrame | None:
    path = Path(path)

    if not path.exists():
        print(f"[WARN] 정유사 weekly prediction 파일 없음: {path}")
        return None

    df = pd.read_csv(path)

    if "week_end" in df.columns:
        df["week_end"] = pd.to_datetime(df["week_end"], errors="coerce")
    elif "date" in df.columns:
        df["week_end"] = pd.to_datetime(df["date"], errors="coerce")
    else:
        raise ValueError(f"정유사 weekly prediction 파일에 week_end/date 컬럼이 없습니다: {path}")

    return df.dropna(subset=["week_end"]).sort_values("week_end").reset_index(drop=True)


def build_refinery_weekly_policy(week_df: pd.DataFrame,
                                 policy_df: pd.DataFrame,
                                 fuel_name: str) -> pd.DataFrame:
    out = week_df[["week_end"]].drop_duplicates().sort_values("week_end").reset_index(drop=True).copy()
    out["week_start"] = out["week_end"] - pd.Timedelta(days=6)

    out["유종"] = fuel_name

    out["유류세정책명"] = pd.NA
    out["유류세인하율_pct"] = 0.0
    out["유류세정책_적용여부"] = False

    out["최고가격제정책명"] = pd.NA
    out["정유사공급가격상한_원L"] = np.nan
    out["최고가격제_적용여부"] = False

    fuel_policy = (
        policy_df[policy_df["유종"] == fuel_name]
        .copy()
        .sort_values(["시작일", "종료일", "카테고리"])
    )

    for _, row in fuel_policy.iterrows():
        mask = (row["시작일"] <= out["week_end"]) & (row["종료일"] >= out["week_start"])

        if int(row["카테고리"]) == 0:
            out.loc[mask, "유류세정책명"] = row["정책명"]
            out.loc[mask, "유류세인하율_pct"] = abs(float(row["가격"]))
            out.loc[mask, "유류세정책_적용여부"] = True

        elif int(row["카테고리"]) == 1:
            out.loc[mask, "최고가격제정책명"] = row["정책명"]
            out.loc[mask, "정유사공급가격상한_원L"] = float(row["가격"])
            out.loc[mask, "최고가격제_적용여부"] = True

    return out


def run_refinery_price_cap_audit(refinery_pred: pd.DataFrame | None,
                                 fuel_name: str,
                                 save_dir: Path) -> pd.DataFrame:
    if refinery_pred is None or len(refinery_pred) == 0:
        return pd.DataFrame()

    save_dir.mkdir(parents=True, exist_ok=True)

    df = refinery_pred.copy()
    policy_weekly = build_refinery_weekly_policy(df[["week_end"]], policy, fuel_name)

    out = df.merge(policy_weekly, on="week_end", how="left")

    actual_col_candidates = ["actual_refinery_pre", "actual_refinery_pre_raw", "dom_level"]
    fair_col_candidates = ["fair_refinery_pre", "pred_actual_refinery_pre", "pred_level"]
    low_col_candidates = ["refinery_band_low", "band_low"]
    high_col_candidates = ["refinery_band_high", "band_high"]

    actual_col = next((c for c in actual_col_candidates if c in out.columns), None)
    fair_col = next((c for c in fair_col_candidates if c in out.columns), None)
    low_col = next((c for c in low_col_candidates if c in out.columns), None)
    high_col = next((c for c in high_col_candidates if c in out.columns), None)

    cap = pd.to_numeric(out["정유사공급가격상한_원L"], errors="coerce")
    tax_rate = pd.to_numeric(out["유류세인하율_pct"], errors="coerce").fillna(0.0) / 100.0
    effective_tax_exvat = BASE_FUEL_TAX_EXVAT_원L[fuel_name] * (1.0 - tax_rate)

    out["유효유류세_exVAT_원L"] = effective_tax_exvat
    out["가격상한_단위주의"] = (
        "정책 CSV 상한은 정유사 공급가격 상한. "
        "정유사 모델은 세전가격이므로 세전 원값과 유류세 가산 추정값을 함께 저장함."
    )

    if actual_col is not None:
        out["정유사_실제세전가격_원L"] = pd.to_numeric(out[actual_col], errors="coerce")
        out["정유사_실제세후추정가격_원L"] = out["정유사_실제세전가격_원L"] + effective_tax_exvat

        out["실제세전가격_상한초과"] = (
            out["최고가격제_적용여부"].fillna(False)
            & cap.notna()
            & (out["정유사_실제세전가격_원L"] > cap)
        )

        out["실제세후추정가격_상한초과"] = (
            out["최고가격제_적용여부"].fillna(False)
            & cap.notna()
            & (out["정유사_실제세후추정가격_원L"] > cap)
        )

        out["실제세후추정_minus_상한_원L"] = out["정유사_실제세후추정가격_원L"] - cap

    if fair_col is not None:
        out["정유사_적정세전가격_원L"] = pd.to_numeric(out[fair_col], errors="coerce")
        out["정유사_적정세후추정가격_원L"] = out["정유사_적정세전가격_원L"] + effective_tax_exvat

        out["적정세전가격_상한초과"] = (
            out["최고가격제_적용여부"].fillna(False)
            & cap.notna()
            & (out["정유사_적정세전가격_원L"] > cap)
        )

        out["적정세후추정가격_상한초과"] = (
            out["최고가격제_적용여부"].fillna(False)
            & cap.notna()
            & (out["정유사_적정세후추정가격_원L"] > cap)
        )

        out["적정세후추정_minus_상한_원L"] = out["정유사_적정세후추정가격_원L"] - cap

    if low_col is not None:
        out["정유사_적정세전범위_하한_원L"] = pd.to_numeric(out[low_col], errors="coerce")
        out["정유사_적정세후추정범위_하한_원L"] = out["정유사_적정세전범위_하한_원L"] + effective_tax_exvat

    if high_col is not None:
        out["정유사_적정세전범위_상한_원L"] = pd.to_numeric(out[high_col], errors="coerce")
        out["정유사_적정세후추정범위_상한_원L"] = out["정유사_적정세전범위_상한_원L"] + effective_tax_exvat

        out["적정세후추정범위상한_상한초과"] = (
            out["최고가격제_적용여부"].fillna(False)
            & cap.notna()
            & (out["정유사_적정세후추정범위_상한_원L"] > cap)
        )

    out["유종"] = fuel_name

    keep_front = [
        "week_end", "week_start", "유종",
        "유류세정책명", "유류세인하율_pct", "유류세정책_적용여부",
        "최고가격제정책명", "정유사공급가격상한_원L", "최고가격제_적용여부",
        "유효유류세_exVAT_원L", "가격상한_단위주의",

        "정유사_실제세전가격_원L",
        "정유사_실제세후추정가격_원L",
        "실제세전가격_상한초과",
        "실제세후추정가격_상한초과",
        "실제세후추정_minus_상한_원L",

        "정유사_적정세전가격_원L",
        "정유사_적정세후추정가격_원L",
        "적정세전가격_상한초과",
        "적정세후추정가격_상한초과",
        "적정세후추정_minus_상한_원L",

        "정유사_적정세전범위_하한_원L",
        "정유사_적정세전범위_상한_원L",
        "정유사_적정세후추정범위_하한_원L",
        "정유사_적정세후추정범위_상한_원L",
        "적정세후추정범위상한_상한초과",
    ]

    cols = [c for c in keep_front if c in out.columns] + [
        c for c in out.columns if c not in keep_front
    ]

    out = out[cols]

    out_path = save_dir / f"정유사_최고가격제_점검_{fuel_name}.csv"
    out.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"[SAVE] {out_path}")

    return out


# =========================================================
# 14) 유종별 실행 함수
# =========================================================
def run_one_fuel(pred_df: pd.DataFrame,
                 actual_col: str,
                 fuel_name: str):
    fuel_dir = OUTPUT_BASE / fuel_name
    plot_dir = fuel_dir / "연도별_정책적용_그래프"

    fuel_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)

    daily_df = build_fuel_daily(
        pred_df=pred_df,
        actual_col=actual_col,
        fuel_name=fuel_name,
    )

    shift_audit = build_policy_shift_audit(daily_df, fuel_name)
    effect_summary = policy_effect_summary(daily_df, fuel_name)
    summary_all = summarize_counts(daily_df, fuel_name)
    summary_year = summarize_counts_by_year(daily_df, fuel_name)

    daily_path = fuel_dir / f"일별_정책적용_데이터_{fuel_name}.csv"
    shift_audit_path = fuel_dir / f"정책시점_감면액감사_{fuel_name}.csv"
    effect_summary_path = fuel_dir / f"유류세_정책효과_요약_{fuel_name}.csv"
    summary_all_path = fuel_dir / f"범위판정_비교요약_{fuel_name}.csv"
    summary_year_path = fuel_dir / f"연도별_범위판정_비교요약_{fuel_name}.csv"

    daily_df.to_csv(daily_path, index=False, encoding="utf-8-sig")
    shift_audit.to_csv(shift_audit_path, index=False, encoding="utf-8-sig")
    effect_summary.to_csv(effect_summary_path, index=False, encoding="utf-8-sig")
    summary_all.to_csv(summary_all_path, index=False, encoding="utf-8-sig")
    summary_year.to_csv(summary_year_path, index=False, encoding="utf-8-sig")

    print(f"[SAVE] {daily_path}")
    print(f"[SAVE] {shift_audit_path}")
    print(f"[SAVE] {effect_summary_path}")
    print(f"[SAVE] {summary_all_path}")
    print(f"[SAVE] {summary_year_path}")

    plot_yearly_policy_applied(
        df=daily_df,
        fuel_name=fuel_name,
        save_dir=plot_dir,
    )

    return daily_df, shift_audit, effect_summary, summary_all, summary_year


print("[READY] 3단계-v2 함수 정의 완료")

### 적용

In [ ]:
# =========================================================
# 3단계-v2 적용 실행
# =========================================================

policy_applicability = build_policy_applicability_report(policy)
policy_applicability.to_csv(
    OUTPUT_BASE / "정책_적용가능여부_요약.csv",
    index=False,
    encoding="utf-8-sig",
)

policy_snapshot = audit_policy_snapshot()
policy_snapshot.to_csv(
    OUTPUT_BASE / "정책_주요일자_감면액_스냅샷.csv",
    index=False,
    encoding="utf-8-sig",
)

gas_daily, gas_shift, gas_effect, gas_summary, gas_summary_year = run_one_fuel(
    pred_df=gas_pred,
    actual_col="보통휘발유_평균",
    fuel_name="휘발유",
)

diesel_daily, diesel_shift, diesel_effect, diesel_summary, diesel_summary_year = run_one_fuel(
    pred_df=diesel_pred,
    actual_col="자동차용경유_평균",
    fuel_name="경유",
)

refinery_cap_audit_all = pd.DataFrame()

# ---------------------------------------------------------
# 전체 요약 저장
# ---------------------------------------------------------
summary_all = pd.concat([gas_summary, diesel_summary], ignore_index=True)
summary_year_all = pd.concat([gas_summary_year, diesel_summary_year], ignore_index=True)
shift_audit_all = pd.concat([gas_shift, diesel_shift], ignore_index=True)
effect_summary_all = pd.concat([gas_effect, diesel_effect], ignore_index=True)

range_count_report_all = pd.concat(
    [
        build_range_count_report(gas_daily, "휘발유"),
        build_range_count_report(diesel_daily, "경유"),
    ],
    ignore_index=True,
)

summary_all.to_csv(
    OUTPUT_BASE / "범위판정_비교요약_전체.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_year_all.to_csv(
    OUTPUT_BASE / "연도별_범위판정_비교요약_전체.csv",
    index=False,
    encoding="utf-8-sig",
)

shift_audit_all.to_csv(
    OUTPUT_BASE / "정책시점_감면액감사_전체.csv",
    index=False,
    encoding="utf-8-sig",
)

effect_summary_all.to_csv(
    OUTPUT_BASE / "유류세_정책효과_요약_전체.csv",
    index=False,
    encoding="utf-8-sig",
)

range_count_report_all.to_csv(
    OUTPUT_BASE / "전체_정책기간_범위판정_요약.csv",
    index=False,
    encoding="utf-8-sig",
)

# ---------------------------------------------------------
# 검증 출력
# ---------------------------------------------------------
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 260)

print("\n[정책 적용 가능/불가 요약]")
print(policy_applicability.to_string(index=False))

print("\n[주요 일자 정책 감면액 스냅샷]")
print(policy_snapshot.to_string(index=False))

print("\n[전체 범위판정 비교요약]")
print(summary_all.to_string(index=False))

print("\n[전체 기준 / 정책 적용 기간 기준 범위판정 요약]")
print(range_count_report_all.to_string(index=False))

print("\n[유류세 정책효과 요약]")
print(effect_summary_all.to_string(index=False))

print("\n[정책시점 감면액 감사]")
print(shift_audit_all.to_string(index=False))


sample_dates = pd.to_datetime([
    "2021-11-11",
    "2021-11-12",
    "2022-07-01",
    "2025-11-01",
    "2026-03-26",
    "2026-03-27",
])

sample_cols = [
    "date",
    "국내유가_원L",
    "적정가격_미정책_원L",
    "적정가격_정책적용_원L",
    "정책효과_원L",
    "유류세인하율_pct",
    "정책감면액_exVAT_원L",
    "정책감면액_VAT포함참고_원L",
    "정책적용_시프트_원L",
    "유류세정책명",
    "최고가격제정책명",
    "정유사공급가격상한_원L",
]

print("\n[휘발유 샘플]")
print(
    gas_daily.loc[
        gas_daily["date"].isin(sample_dates),
        [c for c in sample_cols if c in gas_daily.columns],
    ].to_string(index=False)
)

print("\n[경유 샘플]")
print(
    diesel_daily.loc[
        diesel_daily["date"].isin(sample_dates),
        [c for c in sample_cols if c in diesel_daily.columns],
    ].to_string(index=False)
)


# ---------------------------------------------------------
# 핵심 검증
# ---------------------------------------------------------
print("\n[핵심 검증]")

gas_no_policy_diff = (
    gas_daily["적정가격_미정책_원L"]
    - gas_daily["pred_gross"]
).abs().max()

diesel_no_policy_diff = (
    diesel_daily["적정가격_미정책_원L"]
    - diesel_daily["pred_gross"]
).abs().max()

gas_shift_diff = (
    gas_daily["정책효과_원L"]
    - gas_daily["정책적용_시프트_원L"]
).abs().max()

diesel_shift_diff = (
    diesel_daily["정책효과_원L"]
    - diesel_daily["정책적용_시프트_원L"]
).abs().max()

print("휘발유 미정책가격 - pred_gross max abs:", gas_no_policy_diff)
print("경유 미정책가격 - pred_gross max abs:", diesel_no_policy_diff)
print("휘발유 정책효과 - 시프트 max abs:", gas_shift_diff)
print("경유 정책효과 - 시프트 max abs:", diesel_shift_diff)

print(
    "휘발유 정책적용가격 - 미정책가격 max:",
    (
        gas_daily["적정가격_정책적용_원L"]
        - gas_daily["적정가격_미정책_원L"]
    ).max(),
)

print(
    "경유 정책적용가격 - 미정책가격 max:",
    (
        diesel_daily["적정가격_정책적용_원L"]
        - diesel_daily["적정가격_미정책_원L"]
    ).max(),
)

print(
    "휘발유 정책적용가격 - 미정책가격 min:",
    (
        gas_daily["적정가격_정책적용_원L"]
        - gas_daily["적정가격_미정책_원L"]
    ).min(),
)

print(
    "경유 정책적용가격 - 미정책가격 min:",
    (
        diesel_daily["적정가격_정책적용_원L"]
        - diesel_daily["적정가격_미정책_원L"]
    ).min(),
)

# ---------------------------------------------------------
# 정유사 최고가격제 감사 요약
# ---------------------------------------------------------
if len(refinery_cap_audit_all) > 0:
    print("\n[정유사 최고가격제 감사 요약]")

    bool_cols = [
        "최고가격제_적용여부",
        "실제세전가격_상한초과",
        "실제세후추정가격_상한초과",
        "적정세전가격_상한초과",
        "적정세후추정가격_상한초과",
        "적정세후추정범위상한_상한초과",
    ]

    available_bool_cols = [c for c in bool_cols if c in refinery_cap_audit_all.columns]

    if available_bool_cols:
        print(
            refinery_cap_audit_all
            .groupby(["유종"], dropna=False)[available_bool_cols]
            .sum(numeric_only=True)
            .reset_index()
            .to_string(index=False)
        )

    print("\n[정유사 최고가격제 적용 주간 샘플]")

    refinery_sample_cols = [
        "week_start",
        "week_end",
        "유종",
        "최고가격제정책명",
        "정유사공급가격상한_원L",
        "정유사_실제세전가격_원L",
        "정유사_실제세후추정가격_원L",
        "실제세후추정가격_상한초과",
        "정유사_적정세전가격_원L",
        "정유사_적정세후추정가격_원L",
        "적정세후추정가격_상한초과",
    ]

    cap_sample = refinery_cap_audit_all[
        refinery_cap_audit_all["최고가격제_적용여부"].fillna(False)
    ].copy()

    if len(cap_sample) > 0:
        print(
            cap_sample[
                [c for c in refinery_sample_cols if c in cap_sample.columns]
            ].head(20).to_string(index=False)
        )
    else:
        print("최고가격제 적용 주간이 없습니다. policy CSV의 카테고리 1 기간을 확인하세요.")

else:
    print("\n[정유사 최고가격제 감사 요약]")
    print("정유사 weekly prediction 파일이 없어 최고가격제 감사를 생략했습니다.")


print(f"\n완료: {OUTPUT_BASE}")